[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/composite_strip/composite_strip.ipynb)

# Extension-Twist Coupling in a Composite Strip

This example computes extension-twist coupling in a fibre-reinforced composite strip under axial tension. Each angled ply is much stiffer along its fibres than across them, so stretching along the strip makes the ply shear; when the plies above the midsurface shear opposite to those below, a pure pull twists the strip, whereas reversing the upper half cancels that twist and keeps the strip flat.

Axial strain and dimensionless twist combine into a single even `State` in the strip's cross-sectional plane, and the strain at each height is a duality map from directions to the planes that read their stretch. Turning a ply's fibre direction with a rotor `ply_rotors >> unit_x` lets each ply read strain directly in its own material axes via `fibres & strain_map(fibres)`, without rotating fourth-order stiffness tensors into laminate coordinates. Summing readings against stresses through the thickness builds a quadratic energy map on pairs of states, and relaxing the free long edges subtracts the Schur complement `energy - coupling * coupling / width_stiffness` before `free_width_energy.solve(tension * extension)` finds stretch and twist together.

In the matrix notation of classical lamination theory, as laid out in [Nettles, *Basic Mechanics of Laminated Composite Plates*](https://ntrs.nasa.gov/api/citations/19950009349/downloads/19950009349.pdf), the reduced ply stiffness $Q$ in material coordinates is rotated into laminate coordinates through fourth-power direction cosines to form $\bar Q_{11} = Q_{11}\cos^4\theta + 2(Q_{12} + 2Q_{66})\sin^2\theta\cos^2\theta + Q_{22}\sin^4\theta$, $\bar Q_{16} = (Q_{11} - Q_{12} - 2Q_{66})\sin\theta\cos^3\theta + (Q_{12} - Q_{22} + 2Q_{66})\sin^3\theta\cos\theta$, and the remaining $\bar Q_{ij}$, then integrated piecewise into six-by-six Voigt blocks $(A_{ij}, B_{ij}, D_{ij}) = \int (1, z, z^2)\,\bar Q_{ij}\,dz$ relating force and moment resultants $\begin{pmatrix}N \\ M\end{pmatrix} = \begin{pmatrix}A & B \\ B & D\end{pmatrix}\begin{pmatrix}\varepsilon^0 \\ \kappa\end{pmatrix}$, where axial pull couples to twist through $B_{16}$ after eliminating $\varepsilon^0_y$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.algebras import VGA3D as ga
from numga.extensor import Extensor
from examples.mechanics.composite_strip import render, scenarios
from examples.util.animation import show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Positions and directions in the strip are intrinsic, the duals of the planes x, y and z their unit steps.
Direction = ga.gatype.antivector()                                             # Antivector-
unit_x, unit_y, unit_z = mv.x.dual(), mv.y.dual(), mv.z.dual()                 # [] Direction-, Direction-, Direction-
# Both state components are dimensionless: axial strain and thickness times twist per length.
State = ga.gatype.even().derive.within(Algebra("y+z+"))                        # Rotor+
Plane = ga.gatype.vector()                                                     # Vector+
# Strain is a form on pairs of directions: as a duality map, it sends a direction to the plane that reads its stretch.
Strain = ga.gatype((Plane, Direction, State))                                  # Plane+ <- (Direction-, State+)
extension = mv.scalar([1]).scalar_product(State)                               # [] Scalar+ <- State+
# The minus sign compensates for the negative square of the yz blade.
twist = (-mv.yz).scalar_product(State)                                         # [] Scalar+ <- State+

## Strain readings in each ply's fibre axes

At any height through the thickness, axial extension combines with a twist-induced in-plane shear that reverses sign across the midsurface. Turning the strip axis by each ply's rotor places the fibres directly in that ply's orientation, where pairing against `strain_map` reads longitudinal stretch, transverse stretch, and shear.

In [ ]:
# Two stacks of the same four plies, listed from bottom to top.
angles = scenarios.angles                                                      # [2, 4]
n_cases, n_plies = angles.shape
n_nodes = 2
longitudinal_modulus = scenarios.longitudinal_modulus                          # MPa
transverse_modulus = scenarios.transverse_modulus                              # MPa
poisson, shear_modulus = scenarios.poisson, scenarios.shear_modulus
length, width, thickness = scenarios.length, scenarios.width, scenarios.thickness  # mm
tension = scenarios.tension                                                    # N/mm of width

In [ ]:
def height_strain(height: Scalar, thickness: float) -> Strain:
    """At height `height` (`[...] Scalar+`) through `thickness`, axial extension combines with a
    twist-induced shear of opposite sign above and below the midsurface (`[...] Plane+ <- (Direction-, State+)`)."""
    stretch = mv.x * (mv.x & Direction) * extension                            # [] Plane+ <- (Direction-, State+)
    # Twist tilts the normal both ways in the plane; dividing by thickness gives angle per length.
    shear = -height / thickness * (mv.x * (mv.y & Direction) + mv.y * (mv.x & Direction)) * twist  # [...] Plane+ <- (Direction-, State+)
    return stretch + shear                                                     # [...] Plane+ <- (Direction-, State+)


def ply_readings(strain_map: Extensor, fibres: Direction) -> Extensor:
    """Pairing `strain_map` along `fibres`, across them, and in shear stacks the three ply strain
    readings along a leading axis (`[3, ...]`)."""
    # A quarter turn in the strip's plane gives the in-plane direction perpendicular to the fibres.
    transverse = mv.xy.commutator(fibres)                                      # [...] Direction-
    return stack([fibres & strain_map(fibres), transverse & strain_map(transverse), 2 * (fibres & strain_map(transverse))])


# Two Gauss nodes per ply integrate its quadratic strain energy exactly.
ply_thickness = thickness / n_plies
nodes = np.array([-1, 1]) / np.sqrt(3)                                         # [2]
centres = (np.arange(n_plies) + 0.5) * ply_thickness - thickness / 2           # [n_plies]
heights = unit_z * (centres[:, None] + nodes * ply_thickness / 2)              # [n_plies, n_nodes] Direction-
weights = np.full((n_plies, n_nodes), ply_thickness / 2)                       # [n_plies, n_nodes]
ply_rotors = (mv.xy * (-angles[..., None] / 2)).exp()                          # [n_cases, n_plies, 1] Rotor+
fibres = ply_rotors >> unit_x                                                  # [n_cases, n_plies, 1] Direction-
strain_at_heights = height_strain(mv.z & heights, thickness)                   # [n_plies, n_nodes] Plane+ <- (Direction-, State+)
readings = ply_readings(strain_at_heights, fibres)                             # [3, n_cases, n_plies, n_nodes] Scalar+ <- State+

## Orthotropic ply stresses and free-width response

Poisson contraction couples longitudinal and transverse normal stresses in each ply while shear responds on its own. Because the strip's long edges carry no transverse load, the width contracts until transverse force vanishes, reducing the state stiffness by the Schur complement before axial tension solves for extension and twist.

In [ ]:
def ply_stresses(longitudinal_strain: Extensor, transverse_strain: Extensor, shear_strain: Extensor,
                 longitudinal_modulus: float, transverse_modulus: float, poisson: float,
                 shear_modulus: float) -> tuple[Extensor, Extensor, Extensor]:
    """Orthotropic plane-stress relations couple normal stresses along and across the fibres by
    Poisson contraction while leaving shear stress proportional to `shear_strain`."""
    denominator = 1 - poisson**2 * transverse_modulus / longitudinal_modulus
    longitudinal_stress = (longitudinal_modulus * longitudinal_strain + poisson * transverse_modulus * transverse_strain) / denominator
    transverse_stress = transverse_modulus * (poisson * longitudinal_strain + transverse_strain) / denominator
    return longitudinal_stress, transverse_stress, shear_modulus * shear_strain


def tension_response(fibres: Direction, heights: Direction, weights: np.ndarray, thickness: float,
                     longitudinal_modulus: float, transverse_modulus: float, poisson: float,
                     shear_modulus: float, tension: float) -> tuple[State, Scalar]:
    """Integrating ply strain energy through `thickness` and relaxing the free width yields the
    extension-twist state (`[...] State+`) and transverse strain (`[...] Scalar+`) under axial `tension`.

    `fibres` has shape `[..., n_plies, 1]`, while `heights` and Gauss `weights` have shape
    `[n_plies, n_nodes]`; two Gauss nodes per ply integrate the quadratic strain energy exactly.
    """
    moduli = (longitudinal_modulus, transverse_modulus, poisson, shear_modulus)
    # Each ply reads the strain along its fibres, across them, and as their shear.
    strain_at_heights = height_strain(mv.z & heights, thickness)               # [...] Plane+ <- (Direction-, State+)
    readings = ply_readings(strain_at_heights, fibres)                         # [3, ...] Scalar+ <- State+
    # A unit width strain is read along the same ply axes.
    widening = mv.y * (mv.y & Direction)                                       # [] Plane+ <- Direction-
    width_readings = ply_readings(widening, fibres)                            # [3, ...] Scalar+
    stresses = stack(ply_stresses(*readings, *moduli))                         # [3, ...] Scalar+ <- State+
    width_stresses = stack(ply_stresses(*width_readings, *moduli))             # [3, ...] Scalar+
    # Strain read against stress and summed through the thickness forms the quadratic energy per midsurface area.
    energy = (readings * stresses * weights).sum(axis=(0, -2, -1))             # [...] Scalar+ <- (State+, State+)
    # With the width held fixed, coupling(state) reads the transverse force per unit length.
    coupling = (width_readings * stresses * weights).sum(axis=(0, -2, -1))     # [...] Scalar+ <- State+
    width_stiffness = (width_readings * width_stresses * weights).sum(axis=(0, -2, -1))  # [...] Scalar+
    # Free long edges let the width contract until transverse force vanishes, softening the state energy.
    free_width_energy = energy - coupling * coupling / width_stiffness         # [...] Scalar+ <- (State+, State+)
    # Axial tension drives extension alone; twist settles through the off-diagonal coupling in the energy.
    state = free_width_energy.solve(tension * extension)                       # [...] State+
    width_strain = -coupling(state) / width_stiffness                          # [...] Scalar+
    return state, width_strain


state, width_strain = tension_response(
    fibres, heights, weights, thickness,
    longitudinal_modulus, transverse_modulus, poisson, shear_modulus, tension,
)                                                                              # [n_cases] State+, [n_cases] Scalar+
twist_angle = twist(state) / thickness * length                                # [n_cases] Scalar+

In [ ]:
with np.printoptions(precision=4, suppress=True):
    print("axial strain:     ", extension(state).to_array())
    print("width strain:     ", width_strain.to_array())
    print("twist over length:", np.degrees(twist_angle.to_array()), "degrees")

## Side-by-side deformation under a growing pull

Because the response is linear, scaling the solved state sweeps the strip smoothly from rest to peak tension and back, with displacements magnified fivefold for visibility. Reordering the plies cancels the twist on the right strip and slightly stiffens its axial extension, since twisting gives the antisymmetric stack on the left an extra mode of compliance.

In [ ]:
magnification = scenarios.magnification
fibre_spacing = scenarios.fibre_spacing                                        # mm
separation = scenarios.separation                                              # mm
n_frames, duration_ms = scenarios.n_frames, scenarios.duration_ms
n_length_samples, n_width_samples = scenarios.n_length_samples, scenarios.n_width_samples
loads = scenarios.loads                                                        # [n_frames]

In [ ]:
def midsurface_points(points: Direction, state: State, width_strain: Scalar, thickness: float) -> Direction:
    """Under `state` (`[...] State+`) and `width_strain` (`[...] Scalar+`), `points` (`[..., ...] Direction-`)
    stretch axially, contract across the width, and twist progressively along the strip (`[...] Direction-`)."""
    axial_coord = mv.x & points                                                # [...] Scalar+
    transverse_coord = mv.y & points                                           # [...] Scalar+
    # Each state acts on the whole grid; the two trailing axes sample the midsurface.
    state, width_strain = state[..., None, None], width_strain[..., None, None]  # [..., 1, 1] State+, Scalar+
    # Twist turns each cross-section in the yz plane by an angle proportional to axial position.
    section_rotors = (mv.yz * (-axial_coord * twist(state) / thickness / 2)).exp()  # [...] Rotor+
    cross_sections = section_rotors >> (unit_y * transverse_coord * (1 + width_strain))  # [...] Direction-
    return unit_x * axial_coord * (1 + extension(state)) + cross_sections      # [...] Direction-


def pull_frames(reference: Direction, placement: Direction, state: State, width_strain: Scalar,
                thickness: float, scales: np.ndarray) -> Iterator[Direction]:
    """Yields the displaced midsurface grids (`[...] Direction-`) as the load factor steps through `scales`."""
    for scale in scales:
        yield midsurface_points(reference, state * float(scale), width_strain * float(scale), thickness) + placement


reference = (
    unit_x * np.linspace(0, length, n_length_samples)[:, None]
    + unit_y * np.linspace(-width / 2, width / 2, n_width_samples)[None, :]
)                                                                              # [n_length_samples, n_width_samples] Direction-
# Each strip deforms about its own centreline before sitting beside its counterpart.
placement = unit_y * (separation * np.array([-0.5, 0.5]))[:, None, None]       # [n_cases, 1, 1] Direction-
rest = reference + placement                                                   # [n_cases, n_length_samples, n_width_samples] Direction-
peak = midsurface_points(
    reference, state * magnification, width_strain * magnification, thickness,
) + placement                                                                  # [n_cases, n_length_samples, n_width_samples] Direction-
# Level lines of the coordinate perpendicular to the top ply's fibres trace those fibres on each strip.
top_normal = ply_rotors[:, -1, 0] >> mv.y                                      # [n_cases] Plane+
across_fibres = top_normal[:, None, None] & reference                          # [n_cases, n_length_samples, n_width_samples] Scalar+
recorded_frames = show_states(
    pull_frames(reference, placement, state, width_strain, thickness, magnification * loads),
    render.animate_strips, rest, peak, across_fibres, fibre_spacing,
    name="composite_strip_pull", duration_ms=duration_ms, n_frames=n_frames,
);

Dark strokes mark the top ply's fibres on each tan midsurface as tension builds from the fixed rear edge toward the free front end. The antisymmetric `[-45, -45, 45, 45]` strip on the left twists by roughly seven degrees over its length, while the symmetric `[-45, 45, 45, -45]` strip on the right stretches and narrows in its own plane without twisting.

In [ ]:
# checks
assert readings.shape == (3, n_cases, n_plies, n_nodes)
assert len(recorded_frames) == n_frames
assert ((recorded_frames[0] - rest).antinorm() < 1e-12).all()
assert ((recorded_frames[n_frames // 2] - peak).antinorm() < 1e-12).all()
assert (extension(state) > 0).all()
assert (width_strain < 0).all()
assert twist_angle[0].abs() > 0.1
assert twist_angle[1].abs() < 1e-12
assert extension(state[0]) > extension(state[1])